In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import os
import re
import diffxpy.api as de

In [ ]:
import importlib.util
import sys

def lazy_import(module_name, path_to_file):
    spec = importlib.util.spec_from_file_location(module_name,path_to_file)
    foo = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = foo
    spec.loader.exec_module(foo)
    return foo

utils_dir = "../../utils"
general_utils =  lazy_import("general_utils",os.path.join(utils_dir, "general_utils.py"))

from general_utils import ismember

In [ ]:
# General function for running differential gene expression analysis
def run_diffxpy(
    adata, 
    query_cells, 
    name_reference, 
    name_query, 
    min_cells = 20, 
    reference_cells = [], 
    cell_filtering = [], 
    use_batch = True
):
    output_prefix = "DXP20231027_QUERY_%s_vs_REFERENCE_%s_SIZEFACTOR_%s_BATCH_%s.csv"
    
    if len(reference_cells) == 0:
        reference_cells = np.logical_not(query_cells)
    if len(cell_filtering) == 0:
        cell_filtering = np.logical_or(query_cells, reference_cells)
    adata.obs['factor_to_test'] = ['G2' if x else 'G1' for x in query_cells]
    adata_sub = adata[cell_filtering,:].copy()

    gene_filter = (adata_sub.X > 0).sum(axis=0) > min_cells
    adata_sub = adata_sub[:,gene_filter].copy()

    sub_output_file = output_prefix % (name_query, name_reference, "%s", "%s")
    
    if use_batch:
        formula="~1 + batch + factor_to_test"
        output_file = sub_output_file % ("True", "True")
    else:
        formula="~1 + factor_to_test"
        output_file = sub_output_file % ("True", "False")
        
    # Run differential gene expression
    if(np.logical_not(os.path.exists(os.path.join(output_path, output_file)))):
        print(output_file)
        test = de.test.wald(data=adata_sub, formula_loc=formula, factor_loc_totest='factor_to_test', as_numeric = adata_sub.obs['size_factor'].to_list())
        test.summary().to_csv(os.path.join(output_path, output_file))


Import data and annotations

In [ ]:
input_file = '../data/Reyes_KPLOH_SUBSET_pretumor_PDAC__integrated.h5ad'
output_path = '../differential_expression'
os.makedirs(output_path, exist_ok=True)

# Read adata
adata = sc.read_h5ad(input_file)
adata.X = adata.layers['cellbender'].copy()

df = pd.read_csv('../data/Reyes_KPLOH_SUBSET_pretumor_PDAC__annotated.csv', index_col = 0)
adata.obs_names = [re.sub('-\d+-\d+', '', x) for x in adata.obs_names]

df = df.loc[adata.obs_names,:].copy()
adata.obs['cell_type'] = df['cell_type'].to_numpy()
adata.obs['cell_type_simple'] = df['cell_type_simple'].to_numpy()
adata.obs['karyotype_simple'] = df['karyotype_simple'].to_numpy()
adata.obs['tumor_cells'] = pd.Categorical(df['tumor_cells'].to_numpy())
adata.obs['premalignant_like'] = pd.Categorical(df['premalignant_like'].to_numpy())
adata.obs['tumor_samples'] = pd.Categorical(df['tumor_samples'].to_numpy())
adata.obs['condition_simple'] = pd.Categorical(df['condition_simple'].to_numpy())
adata.obs['batch'] = ["B" + str(x+1) if x == 1 else "B" + str(x) for x in adata.obs['batch']]


Differential gene expression: Progenitor p53 proficient vs Progenitor p53 deficient

In [ ]:
genotype_filter = ismember(adata.obs['condition'], ['K4pretumorDP', 'K4pretumorSP'])[1]
adata_sub = adata[genotype_filter,:].copy()
adata_sub.X = np.array(adata_sub.X.todense())

current_cell_type = 'Progenitor'

name_reference = "DP_diploid_chr6gain_" + current_cell_type
karyotype_filter = ismember(adata_sub.obs['karyotype_simple'], ['Diploid', 'Chr6Gain'])[1]
reference_cells = np.logical_and(adata_sub.obs['cell_type_simple'] == current_cell_type, karyotype_filter)

name_query = "SP_diploid_quiet_" + current_cell_type
karyotype_filter = ismember(adata_sub.obs['karyotype_simple'], ['LOHdiploid', 'LOHquiet'])[1]
query_cells = np.logical_and(adata_sub.obs['cell_type_simple'] == current_cell_type, karyotype_filter)
run_diffxpy(adata_sub, query_cells, name_reference, name_query, reference_cells = reference_cells, use_batch = False)


DXP20231027_QUERY_SP_diploid_quiet_Progenitor_vs_REFERENCE_DP_diploid_chr6gain_Progenitor_SIZEFACTOR_True_BATCH_False.csv
training location model: False
training scale model: True
iter   0: ll=8499839.410764
iter   1: ll=8499839.410764, converged: 0.00% (loc: 100.00%, scale update: False), in 0.00sec
iter   2: ll=3415487.840657, converged: 0.20% (loc: 0.20%, scale update: True), in 10.81sec
iter   3: ll=3415487.840657, converged: 0.20% (loc: 100.00%, scale update: False), in 0.00sec
iter   4: ll=3390023.429610, converged: 97.56% (loc: 97.56%, scale update: True), in 10.90sec
iter   5: ll=3390023.429610, converged: 97.56% (loc: 100.00%, scale update: False), in 0.00sec
iter   6: ll=3386331.814896, converged: 99.43% (loc: 99.43%, scale update: True), in 1.11sec
iter   7: ll=3386331.814896, converged: 99.43% (loc: 100.00%, scale update: False), in 0.00sec
iter   8: ll=3385407.567962, converged: 99.90% (loc: 99.90%, scale update: True), in 0.84sec
iter   9: ll=3385407.567962, converged: 99

/data/lowe/reyesj3/miniconda3/envs/diffxpy/lib/python3.8/site-packages/dask/array/core.py:3138: RuntimeWarning: divide by zero encountered in true_divide
  size = (limit / dtype.itemsize / largest_block) ** (1 / len(autos))


DXP20231027_QUERY_SP_diploid_quiet_Progenitor_vs_REFERENCE_DP_diploid_chr6gain_Progenitor_SIZEFACTOR_True_BATCH_True.csv
training location model: True
training scale model: True
iter   0: ll=83259211.200466
caught 3906 linalg singular matrix errors
iter   1: ll=83244122.749427, converged: 0.00% (loc: 40.30%, scale update: False), in 0.78sec
iter   2: ll=83243626.095067, converged: 0.00% (loc: 40.71%, scale update: False), in 0.71sec
iter   3: ll=83243610.442706, converged: 0.00% (loc: 51.29%, scale update: False), in 0.71sec
iter   4: ll=83243609.368215, converged: 0.00% (loc: 82.87%, scale update: False), in 0.81sec
iter   5: ll=83243609.252457, converged: 0.00% (loc: 95.22%, scale update: False), in 0.64sec
iter   6: ll=30621785.343821, converged: 0.10% (loc: 0.10%, scale update: True), in 11.00sec
caught 3854 linalg singular matrix errors
iter   7: ll=30621088.684411, converged: 0.10% (loc: 46.13%, scale update: False), in 0.76sec
iter   8: ll=30621088.221222, converged: 0.10% (loc:

iter  84: ll=30233201.139252, converged: 99.63% (loc: 99.63%, scale update: True), in 1.01sec
iter  85: ll=30233197.239580, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter  86: ll=30233193.348535, converged: 99.63% (loc: 99.63%, scale update: False), in 0.29sec
iter  87: ll=30233189.470891, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter  88: ll=30233185.605634, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter  89: ll=30233181.749487, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter  90: ll=30233181.594757, converged: 99.63% (loc: 99.63%, scale update: True), in 1.04sec
iter  91: ll=30233177.681998, converged: 99.63% (loc: 99.63%, scale update: False), in 0.43sec
iter  92: ll=30233173.770533, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter  93: ll=30233169.857806, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter  94: ll=30233165.944990, converged: 99.63% (loc

iter 171: ll=30232881.763938, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter 172: ll=30232876.828255, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter 173: ll=30232871.892573, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter 174: ll=30232871.640160, converged: 99.63% (loc: 99.63%, scale update: True), in 1.09sec
iter 175: ll=30232866.602827, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter 176: ll=30232861.565502, converged: 99.63% (loc: 99.63%, scale update: False), in 0.41sec
iter 177: ll=30232856.528199, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter 178: ll=30232851.490951, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter 179: ll=30232846.453854, converged: 99.63% (loc: 99.63%, scale update: False), in 0.28sec
iter 180: ll=30232846.190618, converged: 99.63% (loc: 99.63%, scale update: True), in 1.11sec
iter 181: ll=30232841.047915, converged: 99.63% (loc

...

iter 693: ll=30227421.902255, converged: 99.70% (loc: 99.76%, scale update: False), in 0.24sec
iter 694: ll=30227408.079953, converged: 99.70% (loc: 99.76%, scale update: False), in 0.24sec
iter 695: ll=30227396.307190, converged: 99.70% (loc: 99.77%, scale update: False), in 0.24sec
iter 696: ll=30227308.878219, converged: 99.74% (loc: 99.74%, scale update: True), in 1.78sec
iter 697: ll=30227293.674743, converged: 99.74% (loc: 99.77%, scale update: False), in 0.25sec
iter 698: ll=30227278.779900, converged: 99.74% (loc: 99.77%, scale update: False), in 0.24sec
iter 699: ll=30227264.018055, converged: 99.74% (loc: 99.79%, scale update: False), in 0.24sec
iter 700: ll=30227249.513142, converged: 99.74% (loc: 99.80%, scale update: False), in 0.23sec
iter 701: ll=30227235.554692, converged: 99.74% (loc: 99.80%, scale update: False), in 0.22sec
iter 702: ll=30227142.619175, converged: 99.77% (loc: 99.77%, scale update: True), in 1.67sec
iter 703: ll=30227119.373676, converged: 99.77% (loc

iter 780: ll=30226578.095710, converged: 99.89% (loc: 99.89%, scale update: True), in 2.03sec
iter 781: ll=30226574.693265, converged: 99.89% (loc: 99.89%, scale update: False), in 0.19sec
iter 782: ll=30226571.290976, converged: 99.89% (loc: 99.89%, scale update: False), in 0.19sec
iter 783: ll=30226567.888702, converged: 99.89% (loc: 99.89%, scale update: False), in 0.18sec
iter 784: ll=30226564.486430, converged: 99.89% (loc: 99.89%, scale update: False), in 0.18sec
iter 785: ll=30226561.084158, converged: 99.89% (loc: 99.89%, scale update: False), in 0.19sec
iter 786: ll=30226560.517812, converged: 99.89% (loc: 99.89%, scale update: True), in 2.03sec
iter 787: ll=30226556.881661, converged: 99.89% (loc: 99.89%, scale update: False), in 0.19sec
iter 788: ll=30226553.245694, converged: 99.89% (loc: 99.89%, scale update: False), in 0.18sec
iter 789: ll=30226549.609745, converged: 99.89% (loc: 99.89%, scale update: False), in 0.18sec
iter 790: ll=30226545.973798, converged: 99.89% (loc

iter 867: ll=30226036.101260, converged: 99.94% (loc: 99.94%, scale update: False), in 0.17sec
iter 868: ll=30226033.711915, converged: 99.94% (loc: 99.94%, scale update: False), in 0.18sec
iter 869: ll=30226031.322571, converged: 99.94% (loc: 99.95%, scale update: False), in 0.17sec
iter 870: ll=30225955.701420, converged: 99.94% (loc: 99.94%, scale update: True), in 1.33sec
iter 871: ll=30225953.046678, converged: 99.94% (loc: 99.95%, scale update: False), in 0.17sec
iter 872: ll=30225950.391989, converged: 99.94% (loc: 99.95%, scale update: False), in 0.17sec
iter 873: ll=30225947.737304, converged: 99.94% (loc: 99.95%, scale update: False), in 0.16sec
iter 874: ll=30225945.082624, converged: 99.94% (loc: 99.95%, scale update: False), in 0.16sec
iter 875: ll=30225942.427961, converged: 99.94% (loc: 99.95%, scale update: False), in 0.16sec
iter 876: ll=30225934.246900, converged: 99.94% (loc: 99.94%, scale update: True), in 1.31sec
iter 877: ll=30225931.188994, converged: 99.94% (loc

iter 954: ll=30225653.615565, converged: 99.98% (loc: 99.98%, scale update: True), in 0.14sec
iter 955: ll=30225653.615336, converged: 99.98% (loc: 99.98%, scale update: False), in 0.15sec
iter 956: ll=30225653.615329, converged: 99.98% (loc: 99.98%, scale update: False), in 0.15sec
iter 957: ll=30225653.615329, converged: 99.98% (loc: 99.98%, scale update: False), in 0.15sec
iter 958: ll=30225653.615329, converged: 99.98% (loc: 100.00%, scale update: False), in 0.15sec
iter 959: ll=30225653.615328, converged: 99.98% (loc: 99.98%, scale update: True), in 0.14sec
iter 960: ll=30225653.615328, converged: 99.98% (loc: 100.00%, scale update: False), in 0.15sec
iter 961: ll=30225653.615328, converged: 100.00% (loc: 100.00%, scale update: True), in 0.14sec


/data/lowe/reyesj3/miniconda3/envs/diffxpy/lib/python3.8/site-packages/dask/array/core.py:3138: RuntimeWarning: divide by zero encountered in true_divide
  size = (limit / dtype.itemsize / largest_block) ** (1 / len(autos))


Differential gene expression testing

In [ ]:
### Choose only cell types in pretumor p53 proficient samples
unique_cell_types = np.unique(adata.obs['cell_type_simple'][adata.obs['condition'] == 'K4pretumorDP'])

### Progenitor DP vs Gastric DP
genotype_filter = adata.obs['condition'] == 'K4pretumorDP'
adata_sub = adata[genotype_filter,:].copy()
adata_sub.X = np.array(adata_sub.X.todense()).astype(np.uint16)

current_cell_type = 'Progenitor'
query_cells = adata_sub.obs['cell_type_simple'] == current_cell_type
name_query = "DP_" + current_cell_type

other_cell_type = 'Other'
name_reference = "DP_" + other_cell_type
reference_cells = np.logical_not(adata_sub.obs['cell_type_simple'] == current_cell_type)

run_diffxpy(adata_sub, query_cells, name_reference, name_query, reference_cells = reference_cells, use_batch = True)


DXP20231027_QUERY_DP_Progenitor_vs_REFERENCE_DP_Other_SIZEFACTOR_True_BATCH_False.csv


/data/lowe/reyesj3/miniconda3/envs/diffxpy/lib/python3.8/site-packages/batchglm/models/base_glm/utils.py:110: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  np.mean(x[np.where(grouping == g)[0], :], axis=0)
/data/lowe/reyesj3/miniconda3/envs/diffxpy/lib/python3.8/site-packages/batchglm/models/base_glm/utils.py:158: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks':

training location model: False
training scale model: True
iter   0: ll=182785531.502502
iter   1: ll=182785531.502502, converged: 0.00% (loc: 100.00%, scale update: False), in 0.00sec
iter   2: ll=177194864.703316, converged: 0.14% (loc: 0.14%, scale update: True), in 286.14sec
iter   3: ll=177194864.703316, converged: 0.14% (loc: 100.00%, scale update: False), in 0.00sec
iter   4: ll=177083548.814803, converged: 98.73% (loc: 98.73%, scale update: True), in 275.29sec
iter   5: ll=177083548.814803, converged: 98.73% (loc: 100.00%, scale update: False), in 0.00sec
iter   6: ll=177075237.828155, converged: 99.65% (loc: 99.65%, scale update: True), in 58.78sec
iter   7: ll=177075237.828155, converged: 99.65% (loc: 100.00%, scale update: False), in 0.00sec
iter   8: ll=177074041.509881, converged: 99.91% (loc: 99.91%, scale update: True), in 53.79sec
iter   9: ll=177074041.509881, converged: 99.91% (loc: 100.00%, scale update: False), in 0.00sec
iter  10: ll=177073611.045763, converged: 99.

/data/lowe/reyesj3/miniconda3/envs/diffxpy/lib/python3.8/site-packages/dask/array/core.py:3138: RuntimeWarning: divide by zero encountered in true_divide
  size = (limit / dtype.itemsize / largest_block) ** (1 / len(autos))


DXP20231027_QUERY_DP_Progenitor_vs_REFERENCE_DP_Other_SIZEFACTOR_True_BATCH_True.csv


...

iter 468: ll=664071991.698327, converged: 99.10% (loc: 99.10%, scale update: True), in 60.08sec
iter 469: ll=664071560.261315, converged: 99.10% (loc: 99.10%, scale update: False), in 1.10sec
iter 470: ll=664071128.826492, converged: 99.10% (loc: 99.10%, scale update: False), in 1.27sec
iter 471: ll=664070697.397607, converged: 99.10% (loc: 99.10%, scale update: False), in 1.47sec
iter 472: ll=664070265.984843, converged: 99.10% (loc: 99.10%, scale update: False), in 1.37sec
iter 473: ll=664069834.615839, converged: 99.10% (loc: 99.10%, scale update: False), in 1.15sec
iter 474: ll=664069735.586986, converged: 99.10% (loc: 99.10%, scale update: True), in 59.31sec
iter 475: ll=664069261.405157, converged: 99.10% (loc: 99.10%, scale update: False), in 1.31sec
iter 476: ll=664068787.653453, converged: 99.10% (loc: 99.10%, scale update: False), in 1.39sec
iter 477: ll=664068315.054962, converged: 99.10% (loc: 99.10%, scale update: False), in 1.64sec
iter 478: ll=664067845.483965, converged

iter 554: ll=664040172.989188, converged: 99.10% (loc: 99.10%, scale update: False), in 1.65sec
iter 555: ll=664039675.025567, converged: 99.10% (loc: 99.10%, scale update: False), in 1.51sec
iter 556: ll=664039177.062045, converged: 99.10% (loc: 99.10%, scale update: False), in 1.53sec
iter 557: ll=664038679.098795, converged: 99.10% (loc: 99.10%, scale update: False), in 1.40sec
iter 558: ll=664038626.149060, converged: 99.10% (loc: 99.10%, scale update: True), in 60.22sec
iter 559: ll=664038106.680992, converged: 99.10% (loc: 99.10%, scale update: False), in 1.51sec
iter 560: ll=664037587.215063, converged: 99.10% (loc: 99.10%, scale update: False), in 1.50sec
iter 561: ll=664037067.754941, converged: 99.10% (loc: 99.10%, scale update: False), in 1.37sec
iter 562: ll=664036548.310558, converged: 99.10% (loc: 99.10%, scale update: False), in 1.70sec
iter 563: ll=664036028.908641, converged: 99.10% (loc: 99.10%, scale update: False), in 1.37sec
iter 564: ll=664035970.783832, converged

iter 640: ll=663985508.952443, converged: 99.12% (loc: 99.12%, scale update: False), in 1.53sec
iter 641: ll=663984459.243418, converged: 99.12% (loc: 99.12%, scale update: False), in 1.49sec
iter 642: ll=663984154.983416, converged: 99.12% (loc: 99.12%, scale update: True), in 59.84sec
iter 643: ll=663982978.626021, converged: 99.12% (loc: 99.12%, scale update: False), in 1.33sec
iter 644: ll=663981810.539732, converged: 99.12% (loc: 99.12%, scale update: False), in 1.49sec
iter 645: ll=663980656.965491, converged: 99.12% (loc: 99.12%, scale update: False), in 1.40sec
iter 646: ll=663979522.170001, converged: 99.12% (loc: 99.12%, scale update: False), in 1.55sec
iter 647: ll=663978407.678098, converged: 99.12% (loc: 99.12%, scale update: False), in 1.36sec
iter 648: ll=663978030.809776, converged: 99.12% (loc: 99.12%, scale update: True), in 59.77sec
iter 649: ll=663976805.293430, converged: 99.12% (loc: 99.12%, scale update: False), in 1.67sec
iter 650: ll=663975590.227269, converged

iter 726: ll=663857243.681440, converged: 99.52% (loc: 99.52%, scale update: True), in 55.84sec
iter 727: ll=663857110.664455, converged: 99.52% (loc: 99.52%, scale update: False), in 0.99sec
iter 728: ll=663856980.319814, converged: 99.52% (loc: 99.52%, scale update: False), in 0.88sec
iter 729: ll=663856852.094675, converged: 99.52% (loc: 99.52%, scale update: False), in 1.09sec
iter 730: ll=663856725.795871, converged: 99.52% (loc: 99.52%, scale update: False), in 0.94sec
iter 731: ll=663856600.985250, converged: 99.52% (loc: 99.52%, scale update: False), in 1.07sec
iter 732: ll=663856587.282543, converged: 99.52% (loc: 99.52%, scale update: True), in 55.20sec
iter 733: ll=663856458.341562, converged: 99.52% (loc: 99.52%, scale update: False), in 1.01sec
iter 734: ll=663856329.666167, converged: 99.52% (loc: 99.52%, scale update: False), in 0.97sec
iter 735: ll=663856201.014527, converged: 99.52% (loc: 99.52%, scale update: False), in 0.93sec
iter 736: ll=663856072.364019, converged

/data/lowe/reyesj3/miniconda3/envs/diffxpy/lib/python3.8/site-packages/dask/array/core.py:3138: RuntimeWarning: divide by zero encountered in true_divide
  size = (limit / dtype.itemsize / largest_block) ** (1 / len(autos))
